# Rateio de Sinergia de Turmas e Alunos

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Notebook complementar à auditoria pedagógica: rateia carga horária e alunos entre cursos que compartilham a mesma turma/aula (sinergia).

---


# SGA — Rateio de Sinergia e Alunos

Notebook complementar ao SGA (não substitui nem depende do `04_auditoria_pedagogica_principal.ipynb` nem do `03_inteligencia_estatistica_academica.ipynb`).

**Objetivo:** para cada aula ao vivo do Mapa Pedagógico, identificar:
- Qual curso transmite a disciplina (curso responsável) e qual a ordem dela.
- Quais outros cursos participam da mesma aula por sinergia.
- Quantos alunos de cada curso participante estão matriculados na disciplina.
- O rateio de alunos e horas (decimal e %) entre os cursos participantes.

Usa os MESMOS filtros e a mesma convenção de blocos executivos do `04_auditoria_pedagogica_principal.ipynb` (formato AO VIVO, contrato CLT EAD/CLT STRICTO/RPA-NF, e ESTENDIDA contando nos 2 blocos), para que os totais reconciliem com a tabela "Detalhamento 1.1" do relatório executivo.

**Entradas esperadas na pasta do notebook** (busca por padrão de nome, igual aos outros notebooks do SGA):
- Mapa Pedagógico (`.xlsx`)
- Dados de Matrículas (`.csv`, separador `;`)
- Base de Alunos DW (`.csv`, separador `;`)

**Saída: 2 arquivos Excel, um por período/bloco executivo** —
`SGA_Rateio_Sinergia_Alunos_Modulo_<mod>_Bloco_1a_Ordem_Estendida_<timestamp>.xlsx` e
`..._Bloco_2a_Ordem_Estendida_<timestamp>.xlsx`. Disciplinas ESTENDIDA aparecem nos dois arquivos
(contam nos dois blocos, mesma convenção do Notebook Principal). Cada arquivo tem:
- **1 aba por cluster** — aula transmitida → total de alunos na aula → alunos de cada curso participante → rateio das 2h entre eles (linhas do próprio cluster destacadas em verde, outros cursos da mesma aula em cinza só para contexto).
- Abas de apoio: Leia-me (metodologia completa), Rateio Detalhado (Todos), Consolidado por Curso, Pendencia Responsavel, Pendencia Aluno — todas restritas ao bloco daquele arquivo.

In [ ]:
import re
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import duckdb

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Pasta onde estão as planilhas de entrada e onde a saída será gravada.
# Segue o mesmo princípio dos outros notebooks do SGA: busca por padrão de
# nome, não por nome exato, porque o arquivo muda de mês para mês.
INPUT_DIR = Path(".")
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)

HORAS_POR_AULA = 2.0  # mesma convenção usada no restante do SGA: 1 aula ao vivo = 2h
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


In [ ]:
# ================================================================
# Funções utilitárias (mesmo padrão de normalização usado no restante do SGA)
# ================================================================

def normalize_text(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()


def normalize_key(v) -> str:
    if pd.isna(v):
        return ""
    s = str(v).strip().upper()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize_order(v) -> str:
    s = normalize_key(v)
    if s.startswith("1"):
        return "1ª"
    if s.startswith("2"):
        return "2ª"
    if "EST" in s:
        return "ESTENDIDA"
    return s


def encontrar_arquivo(padroes, pasta: Path = INPUT_DIR):
    """Busca o arquivo mais recente que bate com algum dos padrões glob.
    Mesmo princípio do Notebook Principal: o nome do arquivo muda mês a mês,
    então a busca é por padrão, não por nome exato."""
    candidatos = []
    for p in padroes:
        candidatos.extend(pasta.glob(p))
        candidatos.extend(pasta.glob(p.upper()))
        candidatos.extend(pasta.glob(p.lower()))
    candidatos = sorted(set(candidatos), key=lambda x: x.stat().st_mtime, reverse=True)
    return candidatos[0] if candidatos else None


def col(df: pd.DataFrame, *nomes):
    """Retorna o primeiro nome de coluna, dentre os candidatos, que existe no df."""
    for n in nomes:
        if n in df.columns:
            return n
    return None


In [ ]:
# ================================================================
# Localizar arquivos de entrada
# ================================================================
mapa_path = encontrar_arquivo(["*MAPA*PEDAG*.xlsx", "*mapa*.xlsx"])
matriculas_path = encontrar_arquivo(["*DADOSMATRICULAS*.csv", "*MATRICULA*.csv"])
alunos_dw_path = encontrar_arquivo(["*ALUNOS*DW*.csv", "*BASE*ALUNOS*.csv"])

faltando = [nome for nome, p in [
    ("Mapa Pedagógico (.xlsx)", mapa_path),
    ("Dados de Matrículas (.csv)", matriculas_path),
    ("Base de Alunos DW (.csv)", alunos_dw_path),
] if p is None]

if faltando:
    raise FileNotFoundError(
        "Não encontrei os seguintes arquivos de entrada na pasta do notebook: "
        + ", ".join(faltando)
    )

print("Arquivos localizados:")
print(f"  Mapa Pedagógico ......... {mapa_path}")
print(f"  Dados de Matrículas ..... {matriculas_path}  ({matriculas_path.stat().st_size/1e6:,.1f} MB)")
print(f"  Base de Alunos DW ....... {alunos_dw_path}  ({alunos_dw_path.stat().st_size/1e6:,.1f} MB)")


## 1. Carregar e preparar o Mapa Pedagógico

In [ ]:
# ================================================================
# Carregar o Mapa Pedagógico e montar a base "por curso"
# ================================================================
# Cada disciplina ao vivo pode ter mais de uma linha no Mapa — uma por curso
# que participa daquela aula (sinergia). Aqui mantemos essa granularidade
# "por curso" de propósito (ao contrário do Notebook Principal, que deduplica),
# porque é exatamente o que precisamos para identificar e ratear a sinergia.
#
# Os filtros e a chave de aula abaixo são os MESMOS do Notebook Principal, para
# que o total de horas bata com a tabela "Detalhamento 1.1" do relatório
# executivo: só formato AO VIVO, só contrato CLT EAD/CLT STRICTO/RPA-NF, e a
# aula é identificada pelo NOME da disciplina (não pelo código do catálogo,
# que varia por currículo/curso mesmo quando é a mesma aula).


def encontrar_aba_principal(path: Path) -> str:
    """A aba principal do Mapa segue o padrão '<módulo>.<ano>' (ex.: '53.2026')."""
    xl = pd.ExcelFile(path)
    for nome in xl.sheet_names:
        if re.fullmatch(r"\d{2}\.\d{4}", nome.strip()):
            return nome
    return xl.sheet_names[0]


def escolher_coluna_formato(df: pd.DataFrame) -> str:
    """O Mapa tem mais de uma coluna 'FORMATO'; usamos a que contém o valor 'AO VIVO'."""
    candidatas = [c for c in df.columns if "FORMATO" in normalize_key(c)]
    for c in candidatas:
        if df[c].dropna().astype(str).map(normalize_key).eq("AO VIVO").any():
            return c
    return None


def limpar_contrato(v) -> str:
    s = normalize_key(v)
    if s in {"RPA NF", "RPA/NF", "RPA"}:
        return "RPA/NF"
    if s in {"CLT EAD", "CLT STRICTO", "SOB DEMANDA"}:
        return s
    if s in {"NAO TEM AULA", "NAO TEM", "NSA", ""}:
        return s if s else "NÃO INFORMADO"
    return normalize_text(v).upper()


CONTRATOS_VALIDOS = ["CLT EAD", "CLT STRICTO", "RPA/NF"]

aba_mapa = encontrar_aba_principal(mapa_path)
mapa_raw = pd.read_excel(mapa_path, sheet_name=aba_mapa)
print(f"Aba principal do Mapa: '{aba_mapa}' ({len(mapa_raw)} linhas)")

c_curso = col(mapa_raw, "CURSO")
c_curso_nome = col(mapa_raw, "NOME CURSO")
c_curriculo = col(mapa_raw, "CURRÍCULO")
c_disciplina_cod = col(mapa_raw, "DISCIPLINA")
c_disciplina_nome = col(mapa_raw, "NOME DISCIPLINA")
c_matriz = col(mapa_raw, "MATRIZ")
c_ordem = col(mapa_raw, "ORDEM")
c_dia = col(mapa_raw, "DIA AULA")
c_horario = col(mapa_raw, "HORÁRIO")
c_sinergia = col(mapa_raw, "SINERGIA")
c_cluster = col(mapa_raw, "CLUSTER")
c_modulo = col(mapa_raw, "MÓD", "MODULO", "MÓDULO")
c_contrato = col(mapa_raw, "MODELO CONTRATO")
c_formato = escolher_coluna_formato(mapa_raw)

colunas_essenciais = {
    "CURSO": c_curso, "NOME CURSO": c_curso_nome, "DISCIPLINA": c_disciplina_cod,
    "NOME DISCIPLINA": c_disciplina_nome, "ORDEM": c_ordem, "DIA AULA": c_dia,
    "HORÁRIO": c_horario, "MÓD": c_modulo, "MODELO CONTRATO": c_contrato, "FORMATO (AO VIVO)": c_formato,
}
faltando_cols = [k for k, v in colunas_essenciais.items() if v is None]
if faltando_cols:
    raise KeyError(f"Colunas essenciais não encontradas no Mapa: {faltando_cols}")

base = pd.DataFrame()
base["curso"] = mapa_raw[c_curso].map(normalize_text)
base["curso_nome"] = mapa_raw[c_curso_nome].map(normalize_text)
base["curriculo"] = mapa_raw[c_curriculo].map(normalize_text) if c_curriculo else ""
base["matriz"] = pd.to_numeric(mapa_raw[c_matriz], errors="coerce") if c_matriz else np.nan
base["disciplina_codigo"] = mapa_raw[c_disciplina_cod].map(normalize_text)
base["disciplina_nome"] = mapa_raw[c_disciplina_nome].map(normalize_text)
base["disciplina_key"] = mapa_raw[c_disciplina_nome].map(normalize_key)
base["ordem"] = mapa_raw[c_ordem].map(normalize_order)
base["dia_key"] = mapa_raw[c_dia].map(normalize_key)
base["dia_exibicao"] = mapa_raw[c_dia].map(normalize_text)
base["horario_key"] = mapa_raw[c_horario].astype(str).map(normalize_key)
base["horario_exibicao"] = mapa_raw[c_horario].map(normalize_text)
base["sinergia"] = mapa_raw[c_sinergia].map(normalize_text) if c_sinergia else ""
base["cluster"] = mapa_raw[c_cluster].map(normalize_text) if c_cluster else ""
base["modulo"] = pd.to_numeric(mapa_raw[c_modulo], errors="coerce")
base["contrato"] = mapa_raw[c_contrato].map(limpar_contrato)
base["formato_ao_vivo"] = mapa_raw[c_formato].map(normalize_key).eq("AO VIVO")

# Mesma correção aplicada no Notebook Principal: tokens de erro do Excel na coluna
# CLUSTER (ex.: "#N/D") não são um cluster de verdade — tratamos como "sem
# cluster preenchido" e inferimos pelo nome do curso. Os rótulos devolvidos
# são os MESMOS já usados na coluna CLUSTER do Mapa (grafia/acentuação idêntica),
# para não duplicar cluster por diferença de acento.
CLUSTER_TOKENS_INVALIDOS = {"", "NAN", "NONE", "#N/D", "#N/A", "#NULO!", "#NUL!", "#DIV/0!", "#VALOR!", "#REF!", "#NOME?", "#NÚM!", "#NUM!", "N/D", "N/A"}


def cluster_valido(v) -> bool:
    return str(v).strip().upper() not in CLUSTER_TOKENS_INVALIDOS


def inferir_cluster_por_curso(curso_nome) -> str:
    s = normalize_key(curso_nome)
    if "ENGENHARIA" in s or "ARQUITETURA" in s or "MANUTENCAO INDUSTRIAL" in s or "PRODUCAO INDUSTRIAL" in s:
        return "ENGENHARIAS E ARQUITETURA"
    if "LICENCIATURA" in s or "PEDAGOGIA" in s:
        return "EDUCAÇÃO"
    if any(k in s for k in ["FARMACIA", "BIOMEDICINA", "NUTRICAO", "FISIOTERAPIA", "ENFERMAGEM", "EDUCACAO FISICA", "GERONTOLOGIA"]):
        return "SAÚDE E BEM-ESTAR"
    if any(k in s for k in ["ADMINISTRACAO", "GESTAO", "CIENCIAS CONTABEIS", "ECONOMICAS", "COMERCIO", "LOGISTICA", "MARKETING"]):
        return "GESTÃO E NEGÓCIOS"
    if any(k in s for k in ["SOFTWARE", "SISTEMAS", "REDES", "TECNOLOGIA", "COMPUTACAO"]):
        return "TECNOLOGIA"
    return "OUTROS"


cluster_invalido_mask = ~base["cluster"].map(cluster_valido)
base.loc[cluster_invalido_mask, "cluster"] = base.loc[cluster_invalido_mask, "curso_nome"].map(inferir_cluster_por_curso)
base["cluster"] = base["cluster"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
if cluster_invalido_mask.any():
    print(f"Cluster corrigido por inferência (curso->cluster) em {cluster_invalido_mask.sum()} linhas do Mapa "
          f"(continham token de erro do Excel, ex.: '#N/D').")

# Filtros — os MESMOS do Notebook Principal, para os totais reconciliarem:
# só aula AO VIVO, só contrato válido (CLT EAD / CLT STRICTO / RPA-NF), e
# disciplina/curso/ordem/dia/horário preenchidos (excluindo "NSA" — currículo
# sem oferta ao vivo neste módulo).
valida = (
    base["formato_ao_vivo"].eq(True) &
    base["contrato"].isin(CONTRATOS_VALIDOS) &
    base["disciplina_key"].ne("") &
    base["curso"].ne("") &
    base["ordem"].isin(["1ª", "2ª", "ESTENDIDA"]) &
    base["dia_key"].ne("") & base["dia_key"].ne("NSA") &
    base["horario_key"].ne("") & base["horario_key"].ne("NSA")
)
base = base[valida].copy()

# Chave da aula pelo NOME da disciplina (não pelo código de catálogo) — mesmo
# critério do Principal. O código de catálogo varia por currículo mesmo quando
# é objetivamente a mesma aula ao vivo; o nome é a identidade estável usada em
# todo o SGA para casar Mapa x Registro x Cluster.
base["chave_aula"] = (
    base["disciplina_key"] + "|" + base["ordem"] + "|" + base["dia_key"] + "|" + base["horario_key"]
)

modulo_alvo = int(base["modulo"].dropna().mode().iloc[0])
ano_alvo = int(re.search(r"(\d{4})", aba_mapa).group(1)) if re.search(r"(\d{4})", aba_mapa) else datetime.now().year

# Uma mesma combinação curso+aula pode aparecer 2x no Mapa por causa de currículos
# diferentes do mesmo curso (evolução de matriz curricular) ou variações no código
# de disciplina. Para o rateio isso é a MESMA participação — mantém 1 linha por
# curso por aula (fica a de matriz mais antiga, só para ter um critério
# determinístico; a métrica de rateio não depende disso). O código de disciplina
# específico de cada curso é preservado (necessário para casar com as matrículas).
base_curso_aula = base.sort_values("matriz").drop_duplicates(["chave_aula", "curso"], keep="first").copy()

# Bloco executivo: mesma regra do Principal. Disciplinas de 1ª/2ª ordem entram só
# no bloco correspondente; ESTENDIDA entra nos DOIS blocos (atravessa o módulo
# inteiro), por isso conta 2x na leitura de demanda — não é duplicidade, é a
# convenção de leitura usada em todo o SGA.
def blocos_da_ordem(ordem: str):
    if ordem == "1ª":
        return ["1ª ordem + ESTENDIDA"]
    if ordem == "2ª":
        return ["2ª ordem + ESTENDIDA"]
    if ordem == "ESTENDIDA":
        return ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]
    return []


BLOCOS = ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]

n_cursos_por_aula = base_curso_aula.groupby("chave_aula")["curso"].nunique()
aulas_sinergicas_set = set(n_cursos_por_aula[n_cursos_por_aula > 1].index)

print(f"Módulo identificado: {modulo_alvo} | Ano: {ano_alvo}")
print(f"Aulas ao vivo únicas (após filtros de formato e contrato, igual ao Principal): {base_curso_aula['chave_aula'].nunique()}")
print(f"  - aulas de curso único (sem sinergia): {(n_cursos_por_aula == 1).sum()}")
print(f"  - aulas sinérgicas (2+ cursos simultâneos): {(n_cursos_por_aula > 1).sum()}")
print(f"  - maior nº de cursos numa mesma aula: {n_cursos_por_aula.max()}")
print(f"  - por ordem: " + ", ".join(f"{o}={c}" for o, c in base_curso_aula.drop_duplicates('chave_aula')['ordem'].value_counts().items()))


## 2. Definir o curso responsável por cada aula

In [ ]:
# ================================================================
# Curso responsável por aula
# ================================================================
# Regra combinada (definida e aprovada com o usuário):
#  1) Se exatamente 1 curso da aula está marcado como "Curso Responsável" no
#     campo SINERGIA do Mapa, usa esse curso — é a fonte mais confiável.
#  2) Caso contrário (nenhum marcado, ou mais de um marcado — dado contraditório
#     na fonte), assume responsável o curso com a MATRIZ mais antiga entre os
#     participantes. Em empate, desempata pelo código do curso (determinístico).
# Em qualquer caso o método usado fica registrado, para auditoria.

base_curso_aula["tem_tag_responsavel"] = base_curso_aula["sinergia"].str.contains("Curso Responsável", na=False)

METODO_TAG = "Tag 'Curso Responsável' no Mapa"
METODO_SEM_TAG = "Sem tag única no Mapa -> matriz mais antiga (inferido)"
METODO_TAG_AMBIGUA = "Tag ambígua (mais de 1 'Curso Responsável') -> matriz mais antiga (inferido)"

resp_rows = []
for chave, g in base_curso_aula.groupby("chave_aula"):
    tags = g[g["tem_tag_responsavel"]]
    if len(tags) == 1:
        responsavel = tags.iloc[0]["curso"]
        metodo = METODO_TAG
    else:
        g_validos = g.dropna(subset=["matriz"])
        alvo = g_validos if len(g_validos) else g
        alvo = alvo.sort_values(["matriz", "curso"])
        responsavel = alvo.iloc[0]["curso"] if len(alvo) else g.iloc[0]["curso"]
        metodo = METODO_SEM_TAG if len(tags) == 0 else METODO_TAG_AMBIGUA
    resp_rows.append({
        "chave_aula": chave,
        "curso_responsavel": responsavel,
        "metodo_responsavel": metodo,
        "n_tags_responsavel": len(tags),
    })

resp_df = pd.DataFrame(resp_rows)

resumo_metodo = resp_df[resp_df["chave_aula"].isin(aulas_sinergicas_set)]["metodo_responsavel"].value_counts()
print("Como o curso responsável foi definido (só aulas sinérgicas):")
print(resumo_metodo.to_string())


## 3. Matrículas: whitelist de qualidade e headcount por curso

In [ ]:
# ================================================================
# Matrículas: aplicar a whitelist de qualidade e contar alunos por
# disciplina x curso, usando DuckDB (os arquivos são grandes — ~5 milhões
# de linhas em Matrículas — então filtramos e agregamos em SQL, sem
# carregar o CSV inteiro em memória com pandas).
# ================================================================
# Regra de whitelist definida com o usuário: só entram alunos que também
# aparecem na Base de Alunos DW no mesmo ano/módulo, E cujo código de curso
# bate nas duas bases (evita contar aluno que já trocou de curso, e evita o
# ruído de qualidade que a base de matrículas tem a mais que o DW).

con = duckdb.connect()

query_headcount = f"""
WITH matriculas AS (
    SELECT ALUNO, DISCIPLINA, COD_CURSO_ALUNO_ATUAL AS CURSO
    FROM read_csv_auto('{matriculas_path.as_posix()}', delim=';', header=True, quote='"')
    WHERE ANO = {ano_alvo} AND SEMESTRE = {modulo_alvo}
),
whitelist AS (
    SELECT DISTINCT ALUNO, COD_CURSO AS CURSO
    FROM read_csv_auto('{alunos_dw_path.as_posix()}', delim=';', header=True, quote='"')
    WHERE ANO = {ano_alvo} AND SEMESTRE = {modulo_alvo}
),
matriculas_validadas AS (
    SELECT m.ALUNO, m.DISCIPLINA, m.CURSO
    FROM matriculas m
    INNER JOIN whitelist w ON m.ALUNO = w.ALUNO AND m.CURSO = w.CURSO
)
SELECT DISCIPLINA AS disciplina_codigo, CURSO AS curso, COUNT(DISTINCT ALUNO) AS alunos
FROM matriculas_validadas
GROUP BY DISCIPLINA, CURSO
"""
headcount = con.execute(query_headcount).fetchdf()

query_totais = f"""
SELECT
    (SELECT COUNT(*) FROM read_csv_auto('{matriculas_path.as_posix()}', delim=';', header=True, quote='"')
     WHERE ANO = {ano_alvo} AND SEMESTRE = {modulo_alvo}) AS linhas_matriculas_modulo,
    (SELECT COUNT(*) FROM read_csv_auto('{matriculas_path.as_posix()}', delim=';', header=True, quote='"') m
     INNER JOIN (SELECT DISTINCT ALUNO, COD_CURSO FROM read_csv_auto('{alunos_dw_path.as_posix()}', delim=';', header=True, quote='"')
                 WHERE ANO = {ano_alvo} AND SEMESTRE = {modulo_alvo}) w
     ON m.ALUNO = w.ALUNO AND m.COD_CURSO_ALUNO_ATUAL = w.COD_CURSO
     WHERE m.ANO = {ano_alvo} AND m.SEMESTRE = {modulo_alvo}) AS linhas_apos_whitelist
"""
totais = con.execute(query_totais).fetchdf().iloc[0]

print(f"Matrículas no módulo {modulo_alvo}/{ano_alvo}: {totais['linhas_matriculas_modulo']:,.0f} linhas")
print(f"Após aplicar a whitelist (aluno + curso na Base Alunos DW): {totais['linhas_apos_whitelist']:,.0f} linhas "
      f"({totais['linhas_apos_whitelist']/totais['linhas_matriculas_modulo']:.1%})")
print(f"Combinações disciplina x curso com headcount: {len(headcount):,}")


## 4. Calcular o rateio de alunos e horas por aula (e expandir por bloco executivo)

In [ ]:
# ================================================================
# Rateio por aula: junta headcount aos cursos participantes e calcula
# a proporção (decimal e %) e as horas rateadas de cada curso na aula.
# ================================================================
rateio = base_curso_aula.merge(headcount, on=["disciplina_codigo", "curso"], how="left")
rateio = rateio.merge(resp_df, on="chave_aula", how="left")

rateio["aluno_nao_localizado"] = rateio["alunos"].isna()
rateio["alunos"] = rateio["alunos"].fillna(0).astype(int)
rateio["n_cursos_aula"] = rateio.groupby("chave_aula")["curso"].transform("nunique")
rateio["total_alunos_aula"] = rateio.groupby("chave_aula")["alunos"].transform("sum")
rateio["horas_aula"] = HORAS_POR_AULA

# Curso único na aula: não há com quem ratear, então fica com 100% da hora,
# mesmo que o headcount de matrícula dele não tenha sido localizado (esse caso
# vai para a aba de pendência como aviso, mas não afeta a distribuição de horas).
aula_unica = rateio["n_cursos_aula"].eq(1)
rateio["pct_alunos"] = np.where(
    aula_unica, 1.0,
    np.where(rateio["total_alunos_aula"] > 0, rateio["alunos"] / rateio["total_alunos_aula"], np.nan)
)
rateio["horas_rateadas"] = rateio["pct_alunos"] * rateio["horas_aula"]
rateio["is_sinergica"] = rateio["chave_aula"].isin(aulas_sinergicas_set)

# Conferência: a soma das horas rateadas nunca pode passar da soma das horas plenas
# (2h x nº de aulas únicas) — cada aula continua "custando" só 2h no total, só que
# divididas entre quem participa dela.
horas_esperadas = base_curso_aula["chave_aula"].nunique() * HORAS_POR_AULA
horas_rateadas_total = rateio["horas_rateadas"].sum()
diferenca = abs(horas_rateadas_total - horas_esperadas)
print(f"Conferência de integridade do rateio:")
print(f"  Aulas únicas x {HORAS_POR_AULA:.0f}h = {horas_esperadas:,.1f}h (esperado)")
print(f"  Soma de horas_rateadas em todos os cursos = {horas_rateadas_total:,.1f}h (calculado)")
if diferenca > 0.01:
    print(f"  ATENÇÃO: diferença de {diferenca:.2f}h — revisar aulas sem nenhum headcount localizado.")
else:
    print("  OK — bate exatamente. Nenhuma hora perdida ou duplicada no rateio.")

# ================================================================
# Expandir por bloco executivo (1ª ordem + ESTENDIDA / 2ª ordem + ESTENDIDA)
# ================================================================
# Mesma regra do Notebook Principal (expandir_periodos_operacionais): disciplinas de
# ESTENDIDA entram nos dois blocos, dobrando sua contagem de horas na leitura por
# bloco — é assim que o Principal calcula "Detalhamento 1.1", e é o que faz o
# total por bloco deste notebook reconciliar com aquela tabela.
linhas_bloco = []
for _, r in rateio.iterrows():
    for bloco in blocos_da_ordem(r["ordem"]):
        rr = r.to_dict()
        rr["bloco"] = bloco
        linhas_bloco.append(rr)
rateio_bloco = pd.DataFrame(linhas_bloco)

horas_bloco_total = rateio_bloco.groupby("bloco")["horas_rateadas"].sum()
print()
print("Conferência por bloco executivo (deve bater com 'Detalhamento 1.1' do Principal):")
for bloco in BLOCOS:
    print(f"  {bloco}: {horas_bloco_total.get(bloco, 0):,.1f}h")
print(f"  Total geral: {horas_bloco_total.sum():,.1f}h")


## 5. Consolidar por curso

In [ ]:
# ================================================================
# Consolidado por curso: soma do rateio em todas as aulas em que o curso
# participa, para saber quantas horas efetivas cada curso "puxa" da grade.
# ================================================================
consolidado_curso = rateio.groupby(["curso", "curso_nome"], as_index=False).agg(
    aulas_participadas=("chave_aula", "nunique"),
    aulas_sinergicas_participadas=("is_sinergica", "sum"),
    horas_plenas=("horas_aula", "sum"),
    horas_rateadas=("horas_rateadas", "sum"),
    alunos_total_disciplinas=("alunos", "sum"),
)
resp_count = rateio.loc[rateio["curso"] == rateio["curso_responsavel"]].groupby("curso").size()
consolidado_curso["aulas_como_responsavel"] = consolidado_curso["curso"].map(resp_count).fillna(0).astype(int)

# horas_plenas = quanto custaria se o curso tivesse a aula só para ele.
# horas_rateadas = quanto ele efetivamente "consome" depois do rateio de sinergia.
# A diferença entre as duas é o ganho de eficiência de compartilhar a aula com outros cursos.
consolidado_curso["horas_economizadas_sinergia"] = consolidado_curso["horas_plenas"] - consolidado_curso["horas_rateadas"]
consolidado_curso["pct_economia_sinergia"] = np.where(
    consolidado_curso["horas_plenas"] > 0,
    consolidado_curso["horas_economizadas_sinergia"] / consolidado_curso["horas_plenas"],
    0.0,
)
consolidado_curso = consolidado_curso.sort_values("horas_rateadas", ascending=False).reset_index(drop=True)

total_row = pd.DataFrame([{
    "curso": "TOTAL GERAL", "curso_nome": "",
    "aulas_participadas": consolidado_curso["aulas_participadas"].sum(),
    "aulas_sinergicas_participadas": consolidado_curso["aulas_sinergicas_participadas"].sum(),
    "horas_plenas": consolidado_curso["horas_plenas"].sum(),
    "horas_rateadas": consolidado_curso["horas_rateadas"].sum(),
    "alunos_total_disciplinas": consolidado_curso["alunos_total_disciplinas"].sum(),
    "aulas_como_responsavel": consolidado_curso["aulas_como_responsavel"].sum(),
    "horas_economizadas_sinergia": consolidado_curso["horas_economizadas_sinergia"].sum(),
    "pct_economia_sinergia": np.nan,
}])
consolidado_curso_com_total = pd.concat([consolidado_curso, total_row], ignore_index=True)

print(f"Cursos com participação em aulas ao vivo: {len(consolidado_curso)}")
print(consolidado_curso.head(10).to_string(index=False))


## 6. Tabelas de pendência (auditoria)

In [ ]:
# ================================================================
# Pendências — nada aqui é "escondido" dentro do rateio: tudo que exigiu uma
# regra de fallback ou não pôde ser resolvido vira uma linha auditável.
# ================================================================

# 1) Aulas sinérgicas em que o Mapa não indicou 1 curso responsável com clareza
pendencia_responsavel = (
    resp_df[
        (resp_df["metodo_responsavel"] != METODO_TAG) &
        (resp_df["chave_aula"].isin(aulas_sinergicas_set))
    ]
    .merge(
        base_curso_aula[["chave_aula", "disciplina_nome", "ordem", "dia_exibicao", "horario_exibicao"]]
        .drop_duplicates("chave_aula"),
        on="chave_aula", how="left",
    )
    .merge(
        base_curso_aula.groupby("chave_aula")["curso"].apply(lambda s: ", ".join(sorted(s))).rename("cursos_participantes"),
        on="chave_aula", how="left",
    )
    .sort_values(["metodo_responsavel", "disciplina_nome"])
    .reset_index(drop=True)
)

# 2) Combinações curso/aula sem headcount de matrícula localizado
pendencia_aluno = rateio[rateio["aluno_nao_localizado"]][[
    "chave_aula", "curso", "curso_nome", "disciplina_nome", "ordem",
    "dia_exibicao", "horario_exibicao", "n_cursos_aula", "is_sinergica",
]].sort_values(["is_sinergica", "curso"], ascending=[False, True]).reset_index(drop=True)

print(f"Pendência 1 — responsável não identificado com clareza pelo Mapa: {len(pendencia_responsavel)} aulas sinérgicas")
print(f"Pendência 2 — curso/aula sem matrícula localizada: {len(pendencia_aluno)} linhas "
      f"({pendencia_aluno['is_sinergica'].sum()} delas em aulas sinérgicas — essas SIM afetam o rateio)")


## 7. Exportar Excel (2 arquivos — 1 por bloco executivo — cada um com 1 aba por cluster + abas de apoio)

In [ ]:
# ================================================================
# Exportar Excel — 1 ARQUIVO POR BLOCO EXECUTIVO, cada um com 1 aba por
# cluster + abas de apoio, restritos àquele período.
# ================================================================
# Em cada aba de cluster, a leitura é hierárquica: aula transmitida > total de
# alunos na aula > alunos de cada curso participante > rateio das 2h entre eles.
# Quando uma aula é sinérgica entre cursos de clusters diferentes, ela aparece
# na aba de cada cluster envolvido, mostrando TODOS os participantes (para o
# rateio fazer sentido) — mas o total de horas do cluster soma só a fatia que
# é dele. Disciplinas ESTENDIDA aparecem nos dois arquivos (contam nos dois
# blocos), exatamente como no relatório executivo do Notebook Principal.
import xlsxwriter

SUFIXO_BLOCO = {
    "1ª ordem + ESTENDIDA": "1a_Ordem_Estendida",
    "2ª ordem + ESTENDIDA": "2a_Ordem_Estendida",
}

fmt_title = None  # (definidos dentro de cada workbook, abaixo)


def nome_aba_valido(nome, usados):
    limpo = re.sub(r"[:\\/?*\[\]]", "", nome).strip()
    limpo = limpo[:31] if limpo else "Cluster"
    candidato, i = limpo, 2
    while candidato in usados:
        sufixo = f" ({i})"
        candidato = limpo[: 31 - len(sufixo)] + sufixo
        i += 1
    usados.add(candidato)
    return candidato


COLS_DETALHE = [
    ("Ordem", 8), ("Dia", 10), ("Horário", 10), ("Curso Responsável", 16),
    ("Total Alunos na Aula", 12), ("Curso Participante", 42), ("Cluster do Curso", 22),
    ("Alunos do Curso", 12), ("% do Curso", 11), ("Horas Rateadas", 12),
]


def gerar_excel_do_bloco(bloco_nome: str) -> Path:
    rb = rateio_bloco[rateio_bloco["bloco"] == bloco_nome].copy()

    caminho = OUTPUT_DIR / (
        f"SGA_Rateio_Sinergia_Alunos_Modulo_{modulo_alvo}_Bloco_{SUFIXO_BLOCO[bloco_nome]}_{timestamp}.xlsx"
    )
    wb = xlsxwriter.Workbook(str(caminho))

    fmt_title = wb.add_format({"bold": True, "font_size": 14, "font_color": "#0F172A"})
    fmt_subtitle = wb.add_format({"italic": True, "font_size": 10, "font_color": "#475569"})
    fmt_header = wb.add_format({"bold": True, "bg_color": "#0F172A", "font_color": "white", "border": 1, "valign": "vcenter", "text_wrap": True})
    fmt_aula = wb.add_format({"bold": True, "bg_color": "#DBEAFE", "border": 1})
    fmt_curso_este = wb.add_format({"bg_color": "#ECFDF5", "border": 1})
    fmt_curso_este_pct = wb.add_format({"bg_color": "#ECFDF5", "border": 1, "num_format": "0.0%"})
    fmt_curso_este_hora = wb.add_format({"bg_color": "#ECFDF5", "border": 1, "num_format": "0.00"})
    fmt_curso_outro = wb.add_format({"bg_color": "#F8FAFC", "font_color": "#64748B", "border": 1})
    fmt_curso_outro_pct = wb.add_format({"bg_color": "#F8FAFC", "font_color": "#64748B", "border": 1, "num_format": "0.0%"})
    fmt_curso_outro_hora = wb.add_format({"bg_color": "#F8FAFC", "font_color": "#64748B", "border": 1, "num_format": "0.00"})
    fmt_total = wb.add_format({"bold": True, "bg_color": "#E2E8F0", "border": 1})
    fmt_total_hora = wb.add_format({"bold": True, "bg_color": "#E2E8F0", "border": 1, "num_format": "0.00"})
    fmt_pct = wb.add_format({"num_format": "0.0%"})
    fmt_hora = wb.add_format({"num_format": "0.00"})

    def escrever_aba_cluster(ws, cluster_nome, linhas_cluster, total_horas_cluster):
        aulas_do_cluster = linhas_cluster["chave_aula"].unique()
        cursos_deste_cluster = linhas_cluster.loc[linhas_cluster["cluster"] == cluster_nome, "curso"].nunique()

        ws.write(0, 0, f"CLUSTER: {cluster_nome}", fmt_title)
        ws.write(1, 0, f"Bloco: {bloco_nome}", fmt_subtitle)
        ws.write(2, 0,
                 f"Total de horas do cluster neste bloco (já rateadas): {total_horas_cluster:.2f}h   |   "
                 f"Aulas com participação do cluster: {len(aulas_do_cluster)}   |   "
                 f"Cursos do cluster nessas aulas: {cursos_deste_cluster}",
                 fmt_subtitle)

        row = 4
        for j, (nome_col, largura) in enumerate(COLS_DETALHE):
            ws.write(row, j, nome_col, fmt_header)
            ws.set_column(j, j, largura)
        ws.freeze_panes(row + 1, 0)
        row += 1

        aulas_ordenadas = (
            linhas_cluster.drop_duplicates("chave_aula")
            .sort_values(["disciplina_nome", "ordem", "dia_exibicao", "horario_exibicao"])["chave_aula"]
            .tolist()
        )

        for chave in aulas_ordenadas:
            g = linhas_cluster[linhas_cluster["chave_aula"] == chave].sort_values("alunos", ascending=False)
            primeira = g.iloc[0]
            cab = (
                f"{primeira['disciplina_nome']}"
                + (f"  [SINERGIA: {int(primeira['n_cursos_aula'])} cursos]" if primeira["is_sinergica"] else "")

            )
            ws.write(row, 0, primeira["ordem"], fmt_aula)
            ws.write(row, 1, primeira["dia_exibicao"], fmt_aula)
            ws.write(row, 2, primeira["horario_exibicao"], fmt_aula)
            ws.write(row, 3, primeira["curso_responsavel"], fmt_aula)
            ws.write(row, 4, int(primeira["total_alunos_aula"]), fmt_aula)
            ws.merge_range(row, 5, row, 9, cab, fmt_aula)
            row += 1

            for _, r in g.iterrows():
                eh_deste_cluster = r["cluster"] == cluster_nome
                f_txt = fmt_curso_este if eh_deste_cluster else fmt_curso_outro
                f_pct = fmt_curso_este_pct if eh_deste_cluster else fmt_curso_outro_pct
                f_hora = fmt_curso_este_hora if eh_deste_cluster else fmt_curso_outro_hora
                ws.write(row, 0, "", f_txt)
                ws.write(row, 1, "", f_txt)
                ws.write(row, 2, "", f_txt)
                ws.write(row, 3, "", f_txt)
                ws.write(row, 4, "", f_txt)
                marca = "→ " if eh_deste_cluster else "   "
                ws.write(row, 5, f"{marca}{r['curso_nome']} ({r['curso']})", f_txt)
                ws.write(row, 6, r["cluster"], f_txt)
                ws.write(row, 7, int(r["alunos"]), f_txt)
                ws.write(row, 8, r["pct_alunos"] if pd.notna(r["pct_alunos"]) else 0, f_pct)
                ws.write(row, 9, r["horas_rateadas"] if pd.notna(r["horas_rateadas"]) else 0, f_hora)
                ws.set_row(row, None, None, {"level": 1})
                row += 1

        ws.outline_settings(symbols_below=True, symbols_right=True)

    # ---- 1 aba por cluster, maior para menor em horas (dentro deste bloco) ----
    totais_por_cluster = rb.groupby("cluster")["horas_rateadas"].sum().sort_values(ascending=False)
    nomes_usados = set()
    for cluster_nome, total_horas in totais_por_cluster.items():
        aulas_do_cluster = rb.loc[rb["cluster"] == cluster_nome, "chave_aula"].unique()
        linhas_cluster = rb[rb["chave_aula"].isin(aulas_do_cluster)].copy()
        aba = wb.add_worksheet(nome_aba_valido(cluster_nome.title(), nomes_usados))
        escrever_aba_cluster(aba, cluster_nome, linhas_cluster, total_horas)

    # ---- Abas de apoio, restritas a este bloco ----
    cols_rateio_aula = [
        "chave_aula", "disciplina_codigo", "disciplina_nome", "ordem", "dia_exibicao", "horario_exibicao",
        "cluster", "curso", "curso_nome", "curso_responsavel", "metodo_responsavel", "is_sinergica",
        "n_cursos_aula", "alunos", "total_alunos_aula", "pct_alunos", "horas_aula", "horas_rateadas",
        "aluno_nao_localizado",
    ]
    rateio_export = rb[cols_rateio_aula].sort_values(
        ["cluster", "is_sinergica", "chave_aula", "curso"], ascending=[True, False, True, True]
    ).rename(columns={
        "chave_aula": "Chave da Aula", "disciplina_codigo": "Código Disciplina", "disciplina_nome": "Disciplina",
        "ordem": "Ordem", "dia_exibicao": "Dia", "horario_exibicao": "Horário", "cluster": "Cluster",
        "curso": "Código Curso", "curso_nome": "Curso", "curso_responsavel": "Curso Responsável",
        "metodo_responsavel": "Método (Responsável)", "is_sinergica": "É Sinérgica",
        "n_cursos_aula": "Nº Cursos na Aula", "alunos": "Alunos do Curso", "total_alunos_aula": "Total Alunos na Aula",
        "pct_alunos": "% Alunos", "horas_aula": "Horas da Aula", "horas_rateadas": "Horas Rateadas",
        "aluno_nao_localizado": "Aluno Não Localizado",
    })

    consolidado_bloco = rb.groupby(["curso", "curso_nome"], as_index=False).agg(
        aulas_participadas=("chave_aula", "nunique"),
        aulas_sinergicas_participadas=("is_sinergica", "sum"),
        horas_plenas=("horas_aula", "sum"),
        horas_rateadas=("horas_rateadas", "sum"),
        alunos_total_disciplinas=("alunos", "sum"),
    )
    resp_count_bloco = rb.loc[rb["curso"] == rb["curso_responsavel"]].groupby("curso").size()
    consolidado_bloco["aulas_como_responsavel"] = consolidado_bloco["curso"].map(resp_count_bloco).fillna(0).astype(int)
    consolidado_bloco["horas_economizadas_sinergia"] = consolidado_bloco["horas_plenas"] - consolidado_bloco["horas_rateadas"]
    consolidado_bloco["pct_economia_sinergia"] = np.where(
        consolidado_bloco["horas_plenas"] > 0,
        consolidado_bloco["horas_economizadas_sinergia"] / consolidado_bloco["horas_plenas"], 0.0,
    )
    consolidado_bloco = consolidado_bloco.sort_values("horas_rateadas", ascending=False).reset_index(drop=True)
    total_row_bloco = pd.DataFrame([{
        "curso": "TOTAL GERAL", "curso_nome": "",
        "aulas_participadas": consolidado_bloco["aulas_participadas"].sum(),
        "aulas_sinergicas_participadas": consolidado_bloco["aulas_sinergicas_participadas"].sum(),
        "horas_plenas": consolidado_bloco["horas_plenas"].sum(),
        "horas_rateadas": consolidado_bloco["horas_rateadas"].sum(),
        "alunos_total_disciplinas": consolidado_bloco["alunos_total_disciplinas"].sum(),
        "aulas_como_responsavel": consolidado_bloco["aulas_como_responsavel"].sum(),
        "horas_economizadas_sinergia": consolidado_bloco["horas_economizadas_sinergia"].sum(),
        "pct_economia_sinergia": np.nan,
    }])
    consolidado_export = pd.concat([consolidado_bloco, total_row_bloco], ignore_index=True).rename(columns={
        "curso": "Código Curso", "curso_nome": "Curso", "aulas_participadas": "Aulas Participadas",
        "aulas_sinergicas_participadas": "Aulas Sinérgicas", "aulas_como_responsavel": "Aulas como Responsável",
        "horas_plenas": "Horas Plenas (sem rateio)", "horas_rateadas": "Horas Rateadas",
        "horas_economizadas_sinergia": "Horas 'Economizadas' pela Sinergia",
        "alunos_total_disciplinas": "Soma de Alunos (todas disciplinas)", "pct_economia_sinergia": "% Economia por Sinergia",
    })

    aulas_neste_bloco = set(rb["chave_aula"].unique())
    pendencia_resp_bloco = pendencia_responsavel[pendencia_responsavel["chave_aula"].isin(aulas_neste_bloco)]
    pendencia_resp_export = pendencia_resp_bloco.rename(columns={
        "chave_aula": "Chave da Aula", "curso_responsavel": "Curso Responsável (aplicado)",
        "metodo_responsavel": "Método", "n_tags_responsavel": "Nº Tags 'Curso Responsável' no Mapa",
        "disciplina_nome": "Disciplina", "ordem": "Ordem", "dia_exibicao": "Dia", "horario_exibicao": "Horário",
        "cursos_participantes": "Cursos Participantes",
    })[["Chave da Aula", "Disciplina", "Ordem", "Dia", "Horário", "Cursos Participantes",
        "Curso Responsável (aplicado)", "Método", "Nº Tags 'Curso Responsável' no Mapa"]]

    pendencia_aluno_bloco = pendencia_aluno[pendencia_aluno["chave_aula"].isin(aulas_neste_bloco)]
    pendencia_aluno_export = pendencia_aluno_bloco.rename(columns={
        "chave_aula": "Chave da Aula", "curso": "Código Curso", "curso_nome": "Curso", "disciplina_nome": "Disciplina",
        "ordem": "Ordem", "dia_exibicao": "Dia", "horario_exibicao": "Horário",
        "n_cursos_aula": "Nº Cursos na Aula", "is_sinergica": "É Sinérgica (afeta o rateio)",
    })

    leia_me_linhas = [
        f"Módulo {modulo_alvo} / {ano_alvo} — Bloco: {bloco_nome} — gerado em {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
        "",
        "ESTE ARQUIVO COBRE SÓ 1 BLOCO EXECUTIVO",
        f"Este Excel contém apenas as aulas do bloco '{bloco_nome}'. O outro bloco está em um",
        "arquivo separado (mesmo padrão de nome, trocando o sufixo do bloco). Disciplinas ESTENDIDA",
        "aparecem nos DOIS arquivos — contam nos dois blocos, é a mesma convenção do relatório",
        "executivo do Notebook Principal.",
        "",
        "OBJETIVO",
        "Para cada aula ao vivo do Mapa Pedagógico neste bloco, identificar quais cursos participam por",
        "sinergia, quantos alunos de cada curso estão matriculados, e ratear a hora da aula (2h)",
        "proporcionalmente ao peso de alunos de cada curso.",
        "",
        "ABAS POR CLUSTER (uma para cada cluster)",
        "Para cada aula que toca aquele cluster: a aula transmitida, o total de alunos na aula, e a",
        "lista de cursos participantes com seus alunos, % e horas rateadas. A linha do curso é marcada",
        "com uma seta (->) e fundo verde quando o curso pertence a ESTE cluster; os demais cursos",
        "aparecem em cinza, só para dar contexto de como a hora foi dividida. O total de horas do",
        "cluster (no topo da aba) soma apenas a fatia rateada dos cursos que pertencem a ele — não a",
        "aula inteira — porque uma mesma aula sinérgica pode ter cursos de clusters diferentes",
        "participando ao mesmo tempo. Nesses casos, o total por cluster deste notebook pode diferir em",
        "poucas horas do 'Detalhamento 1.1' do Principal (que atribui a aula inteira a só 1 cluster);",
        "o TOTAL GERAL do bloco bate exatamente com o Principal — a diferença existe só na forma de",
        "dividir entre clusters nessas aulas específicas.",
        "As linhas de cada aula podem ser recolhidas (use o agrupamento/outline à esquerda da planilha).",
        "",
        "COMO A AULA É IDENTIFICADA",
        "Chave = nome da disciplina + ordem + dia + horário (mesmo critério do Notebook Principal). Quando a",
        "mesma chave aparece em mais de uma linha do Mapa com cursos diferentes, é uma aula sinérgica.",
        "Só entram aulas com FORMATO = 'AO VIVO' e MODELO CONTRATO em CLT EAD / CLT STRICTO / RPA-NF —",
        "os mesmos filtros do Principal, para os totais reconciliarem.",
        "",
        "COMO O CLUSTER DE CADA CURSO É DEFINIDO",
        "Vem direto da coluna CLUSTER do Mapa. Quando essa coluna vem com erro de fórmula do Excel",
        "(ex.: '#N/D') ou vazia, o cluster é inferido automaticamente pelo nome do curso.",
        "",
        "COMO O CURSO RESPONSÁVEL É DEFINIDO",
        "1) Se exatamente 1 curso da aula está marcado 'Curso Responsável' no campo SINERGIA do Mapa, usa esse.",
        "2) Caso contrário (nenhum marcado, ou mais de um — dado contraditório na fonte), assume responsável",
        "   o curso com a MATRIZ (ano curricular) mais antiga entre os participantes. Fica registrado na coluna",
        "   'Método (Responsável)' qual dos dois casos foi aplicado, e a aba 'Pendencia Responsavel' lista todos",
        "   os casos em que o fallback foi necessário, para correção na fonte se for o caso.",
        "",
        "COMO O ALUNO É CONTADO (WHITELIST DE QUALIDADE)",
        "A base de Dados de Matrículas tem mais alunos do que a Base de Alunos DW por motivo de qualidade de",
        "dado. Por isso só entram no rateio alunos que aparecem em AMBAS as bases, no mesmo ano/módulo, E cujo",
        "código de curso bate nas duas — evita contar aluno que já trocou de curso ou é ruído da base maior.",
        "",
        "COMO O RATEIO É CALCULADO",
        "% Alunos = alunos do curso na disciplina ÷ total de alunos de todos os cursos participantes na mesma aula.",
        "Horas Rateadas = % Alunos × 2h (a mesma convenção de 2h por aula usada no restante do SGA).",
        "Quando a aula tem 1 só curso, ele fica com 100% das horas — não há com quem ratear.",
        "",
        "PENDÊNCIAS (não são erros silenciosos — são casos para revisão)",
        "- Pendencia Responsavel: aulas sinérgicas em que o Mapa não indicou 1 responsável com clareza.",
        "- Pendencia Aluno: combinações curso/aula sem nenhum aluno localizado nas matrículas. Quando isso",
        "  ocorre numa aula de curso único, não afeta o rateio (o curso já fica com 100%). Quando ocorre numa",
        "  aula sinérgica, o curso em questão fica com 0% do rateio até a matrícula ser localizada/corrigida.",
    ]

    ws_leia = wb.add_worksheet("Leia-me")
    ws_leia.set_column(0, 0, 105)
    ws_leia.write(0, 0, "Metodologia — SGA Rateio de Sinergia e Alunos", wb.add_format({"bold": True, "font_size": 13}))
    for i, linha in enumerate(leia_me_linhas, start=1):
        ws_leia.write(i, 0, linha)

    def escrever_tabela_plana(sheet_name, df, col_larguras=None, cols_pct=(), cols_hora=(), linha_total=False):
        ws = wb.add_worksheet(sheet_name)
        for j, colname in enumerate(df.columns):
            ws.write(0, j, colname, fmt_header)
            largura = (col_larguras or {}).get(colname, max(12, min(38, len(str(colname)) + 4)))
            ws.set_column(j, j, largura)
        for i in range(len(df)):
            for j, colname in enumerate(df.columns):
                val = df.iloc[i, j]
                if colname in cols_pct:
                    ws.write(i + 1, j, val if pd.notna(val) else 0, fmt_pct)
                elif colname in cols_hora:
                    ws.write(i + 1, j, val if pd.notna(val) else 0, fmt_hora)
                else:
                    ws.write(i + 1, j, "" if pd.isna(val) else val)
        ws.freeze_panes(1, 0)
        if linha_total:
            for j, colname in enumerate(df.columns):
                val = df.iloc[-1, j]
                f = fmt_total_hora if colname in cols_hora else fmt_total
                ws.write(len(df), j, "" if pd.isna(val) else val, f)
        return ws

    escrever_tabela_plana(
        "Rateio Detalhado (Todos)", rateio_export,
        col_larguras={"Chave da Aula": 42, "Disciplina": 34, "Curso": 42, "Curso Responsável": 16, "Método (Responsável)": 34},
        cols_pct={"% Alunos"}, cols_hora={"Horas da Aula", "Horas Rateadas"},
    )
    escrever_tabela_plana(
        "Consolidado por Curso", consolidado_export,
        col_larguras={"Curso": 46},
        cols_pct={"% Economia por Sinergia"}, cols_hora={"Horas Plenas (sem rateio)", "Horas Rateadas", "Horas 'Economizadas' pela Sinergia"},
        linha_total=True,
    )
    escrever_tabela_plana("Pendencia Responsavel", pendencia_resp_export, col_larguras={"Cursos Participantes": 60, "Disciplina": 34, "Método": 40})
    escrever_tabela_plana("Pendencia Aluno", pendencia_aluno_export, col_larguras={"Curso": 42, "Disciplina": 34})

    wb.close()
    print(f"[{bloco_nome}] {len(totais_por_cluster)} abas de cluster | total {totais_por_cluster.sum():.1f}h | arquivo: {caminho.resolve()}")
    return caminho


arquivos_gerados = [gerar_excel_do_bloco(bloco) for bloco in BLOCOS]


In [ ]:
print("\n" + "=" * 80)
print(f"SGA - Rateio de Sinergia e Alunos | Módulo {modulo_alvo}/{ano_alvo}")
print("=" * 80)
print(f"Aulas ao vivo analisadas ................ {base_curso_aula['chave_aula'].nunique()}")
print(f"  - sinérgicas (2+ cursos) ............... {len(aulas_sinergicas_set)}")
print(f"  - curso único ........................... {base_curso_aula['chave_aula'].nunique() - len(aulas_sinergicas_set)}")
print(f"Cursos com participação em aulas ao vivo . {len(consolidado_curso)}")
print(f"Pendências — responsável não claro ....... {len(pendencia_responsavel)} aulas")
print(f"Pendências — aluno não localizado ........ {len(pendencia_aluno)} linhas "
      f"({int(pendencia_aluno['is_sinergica'].sum())} afetam o rateio)")
print("-" * 80)
print("Arquivos gerados (1 por bloco executivo):")
for p in arquivos_gerados:
    print(" -", p.resolve())
print("-" * 80)
print(f"Total geral (2 blocos somados): {rateio_bloco['horas_rateadas'].sum():,.1f}h")
print("=" * 80 + "\n")
